# 00 · Raw tick data — and the data-engineering problem

> ⏱️ **~15 min** &nbsp;·&nbsp; 🧭 **SOP:** Phase 1 · Import &nbsp;·&nbsp; 🧩 **Feeds:** ALL editions (every strategy starts from this data)
>
> 🎯 **Goal:** Turn ~24 GB of raw broker text into typed, query-able Parquet — without melting your laptop.
>
> 🔑 **The one thing to remember:** Never load raw ticks into RAM — *stream* them with DuckDB, save Parquet once, work from Parquet forever.

## Notebook 00 in one breath

> **Where we are in the journey.** This is *step zero* of being a quant: before any statistics,
> before any strategy, you have **raw data on disk** and the job of turning ~24 GB of text into
> something you can actually compute on. This notebook is the Python mirror of the project's
> **Phase 1 — Import** (`pipeline/import_data.py`). It maps to §3 of `docs/KENKEM_QUANT_OS.md`.

**What you'll learn (data-engineering, not statistics yet):**
1. What a *tick* is and what our broker's export actually contains.
2. Why you **cannot** `pandas.read_csv` a 7 GB file — the in-memory vs streaming distinction.
3. How **DuckDB** streams a huge CSV and computes on it without blowing up RAM.
4. How we derive `mid` and `spread`, the two numbers everything downstream is built on.
5. Why we rewrite everything to **Parquet** (columnar + compressed + typed) exactly once.

> 🧭 **Mental model.** A quant pipeline is a one-way river:
> `raw CSV → processed Parquet → clean Parquet → bars → features → labels`.
> Each stage is written **once** and never edited in place, so any result is reproducible and
> auditable. Notebook 00 is the very first bend in that river.

In [1]:
# --- Standard setup (run me first) -------------------------------------------------
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
import duckdb                 # streams Parquet/CSV without loading it all into RAM
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _find_root(start: Path) -> Path:
    """Walk up until we see the repo's pipeline/ + data/ — works from any sub-dir."""
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "pipeline").is_dir() and (p / "data").is_dir():
            return p
    raise RuntimeError("repo root (with pipeline/ and data/) not found above " + str(start))

ROOT = _find_root(Path.cwd())
DATA = ROOT / "data"
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))   # so `from pipeline import ...` works (the real Layer-1 code)
con  = duckdb.connect()       # one in-memory DuckDB connection we reuse all notebook

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 130)
pd.set_option("display.float_format", lambda v: f"{v:,.6g}")
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": 0.3})

print("repo root :", ROOT)
print("data dir  :", DATA)

repo root : /Users/tokyotechies/Workspace/KEM/dquants
data dir  : /Users/tokyotechies/Workspace/KEM/dquants/data


## Step 1 · What is a tick? Look at the raw file

A **tick** is one update from the broker: a moment in time with a *bid* (price you can sell at) and
an *ask* (price you can buy at). Our data comes from **MetaTrader 5 (MT5)**, exported from an Exness
account, as a **tab-separated** text file per symbol per year.

The header is `<DATE> <TIME> <BID> <ASK> <LAST> <VOLUME> <FLAGS>`. Let's read the first few rows of
the real 2024 BTCUSD file. We pass `LIMIT 5`, so DuckDB reads only the first handful of rows — it
does **not** load the 4 GB file.

In [2]:
raw_btc_2024 = DATA / "btcusd" / "BTCUSD_ticks_mt5_2024.csv"
print("file exists:", raw_btc_2024.exists(), "| size:", f"{raw_btc_2024.stat().st_size/1e9:.2f} GB")

# read_csv_auto sniffs the schema; tab is the delimiter. LIMIT 5 = read only the top of the file.
con.sql(f"""
    SELECT * FROM read_csv_auto('{raw_btc_2024.as_posix()}', delim='\t')
    LIMIT 5
""").df()

file exists: True | size: 4.26 GB


,<DATE>,<TIME>,<BID>,<ASK>,<LAST>,<VOLUME>,<FLAGS>
0,2024-01-01,00:00:00.036000,"42,250.8","42,273",0,0,6
1,2024-01-01,00:00:00.038000,"42,245.7","42,268.6",0,0,6
2,2024-01-01,00:00:00.266000,"42,242.1","42,265.7",0,0,6
3,2024-01-01,00:00:00.484000,"42,240.4","42,263.4",0,0,6
4,2024-01-01,00:00:00.905000,"42,245.1","42,267.1",0,0,6


Notice two things that are **specific to this feed** (and a classic data-quirk trap):

- **`<LAST>` and `<VOLUME>` are `0` on every row.** This is an FX/CFD feed — there is no centralized
  "last traded price" or real traded volume. *Never use tick volume here.* Later we'll use the
  per-bar **tick count** (how many ticks arrived in a minute) as a volume *proxy*.
- **`<DATE>`** is `YYYY.MM.DD` and **`<TIME>`** is `HH:MM:SS.mmm` (millisecond precision). We'll
  fuse them into a single timestamp.

> 📘 **Concept — tick data vs bars (candles).** Ticks are the raw firehose (millions/day, irregular
> spacing). A *bar* (a.k.a. candle) is a summary of all ticks in a fixed window — open/high/low/close
> over, say, 1 minute. Strategies usually act on bars; we *build* bars from ticks in notebook 02. You
> always start from the finest data you have (ticks) and aggregate **down**, never the reverse.

## Step 2 · The big-data problem — why not just `pandas.read_csv`?

The instinct of every software engineer is `df = pd.read_csv(path)`. On these files that command
will **freeze your machine or crash the kernel.** Here's why, in one number: file size on disk vs the
RAM a DataFrame needs.

In [3]:
raws = sorted((DATA / "btcusd").glob("*.csv")) + sorted((DATA / "xauusd").glob("*.csv"))
rows = []
for p in raws:
    gb = p.stat().st_size / 1e9
    rows.append({"file": p.name, "size_GB": round(gb, 2),
                 "approx_DataFrame_RAM_GB": round(gb * 3, 1)})  # parsed floats ≈ 2-4x raw text
sizes = pd.DataFrame(rows)
print("Total raw on disk:", f"{sizes.size_GB.sum():.1f} GB")
sizes

Total raw on disk: 21.2 GB


,file,size_GB,approx_DataFrame_RAM_GB
0,BTCUSD_ticks_mt5_2024.csv,4.26,12.8
1,BTCUSD_ticks_mt5_2025.csv,7.76,23.3
2,BTCUSD_ticks_mt5_2026.csv,0.76,2.3
3,XAUUSD_ticks_mt5_2024.csv,2.07,6.2
4,XAUUSD_ticks_mt5_2025.csv,3.97,11.9
5,XAUUSD_ticks_mt5_2026.csv,2.43,7.3


A 7 GB CSV becomes **~15–25 GB** once parsed into Python objects (text → 64-bit floats, an index,
object overhead). That exceeds the RAM of most laptops, and even if it fit, you'd re-pay that cost
every time you re-open the notebook.

> 📘 **Concept — in-memory vs streaming.**
> *In-memory* (pandas default) loads the **entire** dataset into RAM, then computes. Simple, but
> bounded by your RAM. *Streaming* (DuckDB, Polars `scan_csv`) reads the file in **chunks**, computes
> as it goes, and keeps only the running result in memory. The file can be 100× your RAM and it still
> works. **Rule for this project: never `pandas.read_csv` the raw ticks. Stream them.**

## Step 3 · DuckDB streams — count millions of ticks without loading them

Let's prove it. We'll count every row in the 2026 BTC file (the smallest, ~0.76 GB) and get its time
span — a full scan — while memory stays flat. DuckDB parses the CSV in a streaming pipeline and only
ever holds the running aggregates.

In [4]:
import time
raw_btc_2026 = DATA / "btcusd" / "BTCUSD_ticks_mt5_2026.csv"

t0 = time.perf_counter()
res = con.sql(f"""
    SELECT count(*) AS n_ticks, min("<DATE>") AS first_date, max("<DATE>") AS last_date
    FROM read_csv_auto('{raw_btc_2026.as_posix()}', delim='\t')
""").fetchone()
print(f"rows           : {res[0]:,}")
print(f"date span      : {res[1]} -> {res[2]}")
print(f"scanned {raw_btc_2026.stat().st_size/1e9:.2f} GB in {time.perf_counter()-t0:.1f}s with flat RAM")

rows           : 14,951,271
date span      : 2026-01-01 -> 2026-06-09
scanned 0.76 GB in 0.5s with flat RAM


That's millions of rows scanned from raw text in a few seconds, on a connection that never held more
than a few megabytes. *This* is why the whole Layer-1 pipeline is DuckDB SQL, not pandas.

> 💡 **For the SOP at scale:** the 2025 BTC file is ~148 M rows / 7.2 GB and the production importer
> (`pipeline/import_data.py`) converts it in a single streaming pass with a 6 GB memory cap. Same
> idea, just bigger.

## Step 4 · Derive the two numbers that matter — `mid` and `spread`

Raw bid/ask aren't quite what we model on. We derive:

- **`mid = (bid + ask) / 2`** — a single, broker-neutral "fair price". We build candles and
  indicators on `mid` so the strategy doesn't depend on which side of the spread we quote.
- **`spread = ask − bid`** — the *instant cost* of a round trip, in price units. For a scalper this
  is the single most important cost number in the entire project.

This is exactly the transform the production importer applies. Let's run it on the first rows.

In [5]:
sample = con.sql(f"""
    SELECT
        strptime("<DATE>" || ' ' || "<TIME>", '%Y.%m.%d %H:%M:%S.%f') AS ts,
        "<BID>"                    AS bid,
        "<ASK>"                    AS ask,
        ("<BID>" + "<ASK>")/2.0    AS mid,
        ("<ASK>" - "<BID>")        AS spread
    FROM read_csv_auto('{raw_btc_2026.as_posix()}', delim='\t',
                       types={{'<DATE>': 'VARCHAR', '<TIME>': 'VARCHAR'}})  -- keep dotted text
    LIMIT 8
""").df()
sample

,ts,bid,ask,mid,spread
0,2026-01-01 00:00:05.830,"87,510.5","87,523.1","87,516.8",12.6
1,2026-01-01 00:00:07.529,"87,503","87,515.6","87,509.3",12.6
2,2026-01-01 00:00:07.551,"87,509.5","87,522.1","87,515.8",12.6
3,2026-01-01 00:00:07.590,"87,513.9","87,526.5","87,520.2",12.6
4,2026-01-01 00:00:10.707,"87,505.8","87,518.4","87,512.1",12.6
5,2026-01-01 00:00:11.404,"87,505.8","87,518.4","87,512.1",12.6
6,2026-01-01 00:00:11.418,"87,498.5","87,511.1","87,504.8",12.6
7,2026-01-01 00:00:12.810,"87,498.3","87,510.9","87,504.6",12.6


> 📘 **Concept — bid, ask, mid, spread (the cost of trading).** You always buy at the (higher) **ask**
> and sell at the (lower) **bid**. The gap is the **spread** — money the broker keeps on every round
> trip, paid *before* your strategy makes a cent. On BTC here the spread is ~$12–20; on a move you're
> trying to capture of ~$30, that's a huge tax. A backtest that ignores spread is **fantasy** — we'll
> hammer this point home in notebook 09. We keep `mid` for *signals* and `spread` for *costs*.

## Step 5 · Why Parquet? Columnar, compressed, typed — convert once, read forever

We stream the raw CSV through that transform **once** and save the result as **Parquet** in
`data/processed/`. After that, every downstream notebook reads Parquet in milliseconds. Let's compare
the same year as CSV vs as the already-built Parquet.

In [6]:
csv_2026 = DATA / "btcusd" / "BTCUSD_ticks_mt5_2026.csv"
pq_2026  = DATA / "processed" / "ticks_btcusd_2026.parquet"
print(f"raw CSV     : {csv_2026.stat().st_size/1e9:.2f} GB")
print(f"Parquet     : {pq_2026.stat().st_size/1e9:.2f} GB  (typed + ZSTD-compressed)")

# Columnar win: ask for ONE column's stats; Parquet reads only that column off disk.
t0 = time.perf_counter()
q = con.sql(f"""
    SELECT count(*) n, round(avg(spread),3) avg_spread, round(median(spread),3) med_spread
    FROM read_parquet('{pq_2026.as_posix()}')
""").df()
print(f"\nspread stats from Parquet in {time.perf_counter()-t0:.2f}s:")
q

raw CSV     : 0.76 GB
Parquet     : 0.18 GB  (typed + ZSTD-compressed)



spread stats from Parquet in 0.15s:


,n,avg_spread,med_spread
0,14951271,11.218,12.6


> 📘 **Concept — columnar storage.** A CSV is **row-oriented**: to read the `spread` column you must
> walk past every other field on every row. Parquet is **column-oriented**: each column is stored
> together, typed, and compressed. Asking for `spread` reads *only* the spread bytes. Add the fact
> that it's pre-parsed (no `"12.6"` → `12.6` conversion) and compressed (ZSTD), and you get the 10–100×
> speedups that make interactive research possible. This is *the* file format of modern data work.

The production importer does more than this toy version: it cross-checks the written row count against
`wc -l` of the source, verifies timestamps parsed (no `NULL`s) and are monotonic, and reports negative
spreads — all in `pipeline/import_data.py`. Here's its real transform, which we just mirrored:

In [7]:
from pipeline import import_data
print(import_data._SELECT_SQL)


    SELECT
        strptime("<DATE>" || ' ' || "<TIME>", '%Y.%m.%d %H:%M:%S.%f') AS ts,
        "<BID>"                          AS bid,
        "<ASK>"                          AS ask,
        ("<BID>" + "<ASK>") / 2.0        AS mid,
        ("<ASK>" - "<BID>")              AS spread,
        "<FLAGS>"                        AS flags
    FROM read_csv(
        ?,
        delim = '	',
        header = true,
        columns = {columns}
    )



## Step 6 · Two markets — BTCUSD and XAUUSD

KenKem trades two instruments: **BTCUSD** (crypto, ~24/7) and **XAUUSD** (gold, ~23/5, closes
weekends). Both are imported. Note where each one is in the river right now — BTC is processed all the
way to features; XAU is imported but not yet cleaned/barred (a real-world "to-do" state).

In [8]:
proc = DATA / "processed"
def stage_report(symbol):
    imported = [p for p in proc.glob(f"ticks_{symbol}_2*.parquet") if "clean" not in p.name]
    return {
        "symbol": symbol.upper(),
        "raw_csv_years":  len(list((DATA/symbol).glob("*.csv"))),
        "imported_ticks": len(imported),
        "clean_ticks":    len(list(proc.glob(f"ticks_{symbol}_*_clean.parquet"))),
        "bar_files":      len(list(proc.glob(f"bars_{symbol}_*"))),
        "feature_files":  len(list((DATA/'features').glob(f"features_{symbol}_*"))),
    }
pd.DataFrame([stage_report("btcusd"), stage_report("xauusd")]).set_index("symbol")

,raw_csv_years,imported_ticks,clean_ticks,bar_files,feature_files
symbol,,,,,
BTCUSD,3,3,3,6,2
XAUUSD,3,3,0,0,0


> 🧭 **Why keep the raw files immutable?** The whole pipeline is *one-directional*: we derive new
> files forward and never edit upstream artifacts. If a bug is found in bar-building, we re-run from
> the untouched ticks and get a byte-identical result. This reproducibility is what makes a backtest
> trustworthy — and it's the same discipline as an idempotent data pipeline in software.

## 🎯 Your turn

1. **Peek at XAU.** Change `raw_btc_2024` in Step 1 to `DATA/"xauusd"/"XAUUSD_ticks_mt5_2024.csv"`
   and read the first 5 rows. What is the typical *spread* on gold vs on BTC? (Gold trades near
   $2,000–2,400; its spread is a few cents, but as a *fraction* of price — how does it compare?)
2. **Count gold.** Re-run Step 3's count on a XAU file. How many ticks/year does gold have vs BTC?
3. **Spread by year.** Run the Step 5 spread query on `ticks_btcusd_2024.parquet`,
   `..._2025.parquet`, `..._2026.parquet`. The medians differ a lot — that's the **spread-realism
   trap** we tackle in notebook 01. Write down the three medians.
4. **Prove the leak.** Try `pd.read_csv(raw_btc_2026, sep="\t", nrows=1000)`. It works with
   `nrows=1000`. Now *imagine* removing `nrows` — and don't actually run it. Why not?

➡️ **Next:** notebook **01** — the data isn't clean yet. Bad ticks are the #1 source of fake edges.